# Concatenate all tasks per subject per device

For each (subject, device) pair, appends all task CSVs into one file.

**Input:** `~/fog_labeled_data/IMU_devX_FOGXXX_taskname_labeled_100Hz.csv`

**Output:** `~/fog_labeled_data/combined/IMU_devX_FOGXXX_all_tasks.csv`

In [ ]:
import pandas as pd
from pathlib import Path
from collections import defaultdict

In [ ]:
INPUT_DIR  = Path.home() / 'fog_labeled_data'
OUTPUT_DIR = INPUT_DIR / 'combined'
OUTPUT_DIR.mkdir(exist_ok=True)

# Find all labeled CSVs
all_files = sorted(INPUT_DIR.glob('IMU_dev*_labeled_100Hz.csv'))
print(f'Found {len(all_files)} files')

# Group by (device, subject)
groups = defaultdict(list)
for f in all_files:
    # e.g. IMU_dev2_FOG002_8_Shape_Circuit_1_labeled_100Hz.csv
    parts = f.stem.split('_')  # ['IMU', 'dev2', 'FOG002', '8', 'Shape', ...]
    dev     = parts[1]         # dev2
    fog_id  = parts[2]         # FOG002
    groups[(dev, fog_id)].append(f)

print(f'Groups (device, subject): {len(groups)}')

In [ ]:
summary = []

for (dev, fog_id), files in sorted(groups.items()):
    dfs = []
    for f in sorted(files):
        df = pd.read_csv(f)
        # task_name column is already in the CSV from the pipeline
        # but double-check and fill from filename if missing
        if 'task_name' not in df.columns:
            task = f.stem.replace(f'IMU_{dev}_{fog_id}_', '').replace('_labeled_100Hz', '')
            df['task_name'] = task
        dfs.append(df)

    combined = pd.concat(dfs, ignore_index=True)

    out_path = OUTPUT_DIR / f'IMU_{dev}_{fog_id}_all_tasks.csv'
    combined.to_csv(out_path, index=False)

    fog_pct = 100 * combined['fog_label'].mean()
    tasks   = combined['task_name'].unique().tolist()
    print(f'{fog_id} | {dev} | {len(combined):>7} samples | FOG: {fog_pct:.2f}% | tasks: {len(tasks)}')

    summary.append({
        'subject': fog_id, 'device': dev,
        'n_samples': len(combined),
        'fog_samples': int(combined['fog_label'].sum()),
        'fog_pct': round(fog_pct, 2),
        'n_tasks': len(tasks),
        'tasks': ', '.join(sorted(tasks))
    })

print(f'\n✓ {len(groups)} combined files saved to {OUTPUT_DIR}')

In [ ]:
# Summary table
df_summary = pd.DataFrame(summary)
df_summary.to_csv(OUTPUT_DIR / 'combined_summary.csv', index=False)

print('=== FOG % per subject (pivot by device) ===')
pivot = df_summary.pivot_table(
    index='subject', columns='device',
    values='fog_pct', aggfunc='mean'
).round(2)
print(pivot.to_string())

print('\n=== Total FOG samples per subject ===')
totals = df_summary.groupby('subject')[['fog_samples', 'n_samples']].sum()
totals['fog_pct'] = (100 * totals['fog_samples'] / totals['n_samples']).round(2)
print(totals.to_string())